**代码演示部分：配合本章学习材料使用**  
### 第一部分：张量运算示例  
这里将演示Tensor的一些基本操作

In [ ]:
import torch

In [ ]:
?torch.tensor

In [ ]:
# 创建tensor，用dtype指定类型。注意类型要匹配
a = torch.tensor(1.0, dtype=torch.float)
b = torch.tensor(1, dtype=torch.long)
c = torch.tensor(1.0, dtype=torch.int8)
print(a, b, c)

In [ ]:
# 使用指定类型函数随机初始化指定大小的tensor
d = torch.FloatTensor(2,3)
e = torch.IntTensor(2)
f = torch.IntTensor([1,2,3,4])  #对于python已经定义好的数据结构可以直接转换
print(d, '\n', e, '\n', f)

In [ ]:
# tensor和numpy array之间的相互转换
import numpy as np

g = np.array([[1,2,3],[4,5,6]])

h = torch.tensor(g)
print('h',h)

i = torch.from_numpy(g)
print('i',i)

f=torch.as_tensor(g)
print('f',f)

j = h.numpy()
print('j',j)

注意：torch.tensor创建得到的张量和原数据是不共享内存的，张量对应的变量是独立变量。  
而torch.from_numpy()和torch.as_tensor()从numpy array创建得到的张量和原数据是共享内存的，张量对应的变量不是独立变量，修改numpy array会导致对应tensor的改变。

In [ ]:
g[0,0] = 100
print(i)
print(f)

In [ ]:
# 常见的构造Tensor的函数
k = torch.rand(2, 3) 
l = torch.ones(2, 3)
m = torch.zeros(2, 3)
n = torch.arange(0, 10, 2)
print(k, '\n', l, '\n', m, '\n', n)

In [ ]:
# 查看tensor的维度信息（两种方式）
print(k.shape)
print(k.size())

两者的区别很小，实际效果几乎一样：

**`k.shape`** — 是 tensor 对象的一个**属性**，直接访问，返回 `torch.Size` 对象。

**`k.size()`** — 是 tensor 对象的一个**方法**，调用时可以传参数：
```python
k.size()     # 返回完整的 torch.Size，和 k.shape 一样
k.size(0)    # 返回第 0 维的大小，返回一个整数
k.size(1)    # 返回第 1 维的大小，返回一个整数
```

两者返回的 `torch.Size` 本质上是一个整数元组，行为和 Python tuple 完全一致：
```python
k = torch.rand(2, 3)
k.shape       # torch.Size([2, 3])
k.size()      # torch.Size([2, 3])
k.shape[0]    # 2
k.size(0)     # 2  （更简洁）
```

**实际使用中：**
- 查看完整形状：两者都很常见，`.shape` 略多，和 NumPy 风格一致
- 取某一维的大小：`.size(dim)` 比 `.shape[dim]` 更简洁，但两者都有人用

In [ ]:
# tensor的运算
o = torch.add(k,l)
print(o)

In [ ]:
# tensor的索引方式与numpy类似
o = torch.rand(2, 3,5)

In [ ]:
print(o[0,:,0])

In [ ]:
o = torch.rand(2,3,5)
print(o)
print(o[:,1,])
print(o[0,:])

In [ ]:
# 改变tensor形状的神器：view
print(o.view((3,2)))
print(o.view(-1,2))

In [ ]:
# tensor的广播机制（使用时要注意这个特性）
p = torch.arange(1, 3).view(1, 2)
print(p)
q = torch.arange(1, 4).view(3, 1)
print(q)
print(p + q)

squeeze 和 unsqueeze 是一对相反的操作，专门用来增加或删除大小为 1 的维度。

unsqueeze：在指定位置插入一个大小为 1 的新维度

In [ ]:
print(o)
r = o.unsqueeze(1) # 在第 1 维的位置插入一个新维度
print(r)
print(r.shape)  # torch.Size([2, 1, 3])

squeeze：删除大小为 1 的维度

In [ ]:
s = r.squeeze(0)   # 尝试删除第 0 维
# 第 0 维大小是 2，不是 1，所以不删，shape不变 → [2, 1, 3]
print(s) 
print(s.shape)

In [ ]:
t = r.squeeze(1) 
# 第 1 维是 1，可以删除 → shape 变成 [2, 3]
print(t)
print(t.shape)

### 第二部分：自动求导示例
这里将通过一个简单的函数  $y=x_1+2*x_2$  来说明PyTorch自动求导的过程

In [ ]:
import torch

x1 = torch.tensor(1.0, requires_grad=True)
x2 = torch.tensor(2.0, requires_grad=True)
y = x1 + 2*x2
print(y)

In [ ]:
# 首先查看每个变量是否需要求导
print(x1.requires_grad)
print(x2.requires_grad)
print(y.requires_grad)

In [ ]:
# 查看每个变量导数大小。此时因为还没有反向传播，因此导数都不存在
print(x1.grad.data)
print(x2.grad.data)
print(y.grad.data)

In [ ]:
x1

In [ ]:
## 反向传播后看导数大小
y = x1 + 2*x2
y.backward()
print(x1.grad.data)
print(x2.grad.data)

In [ ]:
# 导数是会累积的，重复运行相同命令，grad会增加
y = x1 + 2*x2
y.backward()
print(x1.grad.data)
print(x2.grad.data)

In [ ]:
# 所以每次计算前需要清除当前导数值避免累积，这一功能可以通过pytorch的optimizer实现。后续会讲到

In [ ]:
# 尝试，如果不允许求导，会出现什么情况？
x1 = torch.tensor(1.0, requires_grad=False)
x2 = torch.tensor(2.0, requires_grad=False)
y = x1 + 2*x2
y.backward()